# End-to-End Sales Analytics Project — Data Cleaning
End-to-end cleaning pipeline: raw tables are pulled from SQL Server, audited for data-quality issues, cleaned with pandas, and pushed back to SQL Server as clean tables ready for the star schema / Power BI layer.

**Objectives:** Clean Raw data for downstream analysis in Power BI    
**Tools:** Python- Pandas, Numpy  
**Dataset:** Northwind Traders  

---
## 1. Setup — Connect to SQL Server and Pull Raw Data

In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import pyodbc
from sqlalchemy import create_engine

# Load Database credentials from .env
load_dotenv()

# Build the connection string
conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={os.getenv('DB_SERVER')};"
    f"DATABASE={os.getenv('DB_NAME')};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# 'mssql+pyodbc://' tells SQLAlchemy to use pyodbc as the driver
connection_url = f"mssql+pyodbc:///?odbc_connect={conn_str}"
engine = create_engine(connection_url)

r_customers = pd.read_sql("SELECT * FROM raw_customers", engine)
r_employees = pd.read_sql("SELECT * FROM raw_employees", engine)
r_order_details = pd.read_sql("SELECT * FROM raw_order_details", engine)
r_orders = pd.read_sql("SELECT * FROM raw_orders", engine)
r_products = pd.read_sql("SELECT * FROM raw_products", engine)

---
## 2. Customers Table

In [2]:
r_customers.head()

,customerID,companyName,contactName,contactTitle,city,country
0,ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Berlin,GERMANY
1,ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Mexico City,Mexico
2,ANTON,Antonio Moreno Taquería,Antonio Moreno,Owner,MEXICO CITY,Mexico
3,AROUT,Around the Horn,Thomas Hardy,Sales Representative,london,UK
4,BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Luleå,Sweden


In [3]:
# Intial Audit - Checking Duplicate Rows, Lowercase, Uppercase, Missing Values.
print(f"Duplicate rows: {r_customers.duplicated('customerID').sum()}")
sample_cust = r_customers['customerID'].head(3).tolist()
print(f"Customers - IDs found: {sample_cust}\nLength of first ID: {len(str(sample_cust[0]))} (Expected 5 if no spaces)")
print(f"Customers - Lowercase cities count: {r_customers['city'].str.islower().sum()}")
print(f"Customers - Uppercase cities count: {r_customers['city'].str.isupper().sum()}")
print(f"Customers - Missing values in the table: {r_customers.isnull().sum().sum()}")
print(f"Customers - String 'NULL' values hidden as text: {(r_customers == 'NULL').sum().sum()}")

Duplicate rows: 0
Customers - IDs found: ['ALFKI', 'ANATR ', 'ANTON']
Length of first ID: 5 (Expected 5 if no spaces)
Customers - Lowercase cities count: 14
Customers - Uppercase cities count: 7
Customers - Missing values in the table: 0
Customers - String 'NULL' values hidden as text: 0


In [4]:
# Fix: clear hidden whitespace and standardize text casing to Title Case
cust_text_cols = ['customerID', 'companyName', 'contactName', 'contactTitle', 'city', 'country']

for col in cust_text_cols:
    r_customers[col] = r_customers[col].astype(str).str.strip()
    if col in ['city', 'country', 'contactName', 'contactTitle']:
        r_customers[col] = r_customers[col].str.title()

print("Customers table cleaned successfully.")
r_customers.head()

Customers table cleaned successfully.


,customerID,companyName,contactName,contactTitle,city,country
0,ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Berlin,Germany
1,ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Mexico City,Mexico
2,ANTON,Antonio Moreno Taquería,Antonio Moreno,Owner,Mexico City,Mexico
3,AROUT,Around the Horn,Thomas Hardy,Sales Representative,London,Uk
4,BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Luleå,Sweden


---
## 3. Employees Table

In [5]:
r_employees.head(20)

,employeeID,employeeName,title,city,country,reportsTo
0,1,Nancy Davolio,Sales Representative,New York,USA,8.0
1,2,Andrew Fuller,Vice President Sales,New York,USA,NULL
2,3,Janet Leverling,Sales Represntative,New York,USA,8.0
3,4,Margaret Peacock,Sales Representative,New York,USA,8.0
4,5,Steven Buchanan,Sales Manager,London,UK,2.0
5,6,Michael Suyama,Sales Representative,London,UK,5.0
6,7,Robert King,Sales Representative,London,UK,5.0
7,8,Laura Callahan,Sales Manager,New York,USA,2.0
8,9,Anne Dodsworth,Sales Represntative,London,UK,5.0


In [6]:
r_employees.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   employeeID    9 non-null      str  
 1   employeeName  9 non-null      str  
 2   title         9 non-null      str  
 3   city          9 non-null      str  
 4   country       9 non-null      str  
 5   reportsTo     9 non-null      str  
dtypes: str(6)
memory usage: 564.0 bytes


In [7]:
# Intial Audit - Checking Duplicate Rows, Missing Values, Typos, Unique titles.
print(f"Duplicate rows (employeeID): {r_employees.duplicated('employeeID').sum()}")
print(f"Employees - Missing values detected as actual NaN: {r_employees.isna().sum().sum()}")
print(f"Employees - String 'NULL' values hidden: {(r_employees == 'NULL').sum().sum()}")
print(f"Employees - Typos found in job titles ('Sales Represntative'): {(r_employees['title'] == 'Sales Represntative').sum()}")
print(f"Unique titles: {r_employees['title'].unique()}")

Duplicate rows (employeeID): 0
Employees - Missing values detected as actual NaN: 0
Employees - String 'NULL' values hidden: 1
Employees - Typos found in job titles ('Sales Represntative'): 2
Unique titles: <StringArray>
['Sales Representative', 'Vice President Sales',  'Sales Represntative',
        'Sales Manager']
Length: 4, dtype: str


In [8]:
# Checking Null value in each Columns.
print(f"Employees - String 'NULL' values hidden :\n{(r_employees == 'NULL').sum()}")

Employees - String 'NULL' values hidden :
employeeID      0
employeeName    0
title           0
city            0
country         0
reportsTo       1
dtype: int64


In [9]:
# Fix: hidden whitespace , casing, job-title typo, ID/reportsTo types
for col in ['employeeName', 'title', 'city', 'country']:
    r_employees[col] = r_employees[col].astype(str).str.strip().str.title()

# Resolve common typo in job titles
r_employees['title'] = r_employees['title'].replace('Sales Represntative', 'Sales Representative')

# Restore proper numeric type for employeeID
r_employees['employeeID'] = pd.to_numeric(r_employees['employeeID'], errors='coerce').astype(int)

# Convert literal string 'NULL' back into actual NaN, then to a nullable integer type
# (Int64 preserves both integers and nulls together, unlike plain int/float)
r_employees['reportsTo'] = r_employees['reportsTo'].replace('NULL', np.nan)
r_employees['reportsTo'] = pd.to_numeric(r_employees['reportsTo'], errors='coerce').astype('Int64')

print("Employees table cleaned successfully.")
r_employees.info()

Employees table cleaned successfully.
<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   employeeID    9 non-null      int64
 1   employeeName  9 non-null      str  
 2   title         9 non-null      str  
 3   city          9 non-null      str  
 4   country       9 non-null      str  
 5   reportsTo     8 non-null      Int64
dtypes: Int64(1), int64(1), str(4)
memory usage: 573.0 bytes


In [10]:
r_employees.head()

,employeeID,employeeName,title,city,country,reportsTo
0,1,Nancy Davolio,Sales Representative,New York,Usa,8
1,2,Andrew Fuller,Vice President Sales,New York,Usa,<NA>
2,3,Janet Leverling,Sales Representative,New York,Usa,8
3,4,Margaret Peacock,Sales Representative,New York,Usa,8
4,5,Steven Buchanan,Sales Manager,London,Uk,2


---
## 4. Order Details Table

In [11]:
r_order_details.head(10)

,orderID,productID,unitPrice,quantity,discount
0,10248,11,14.0,12,0.00
1,10248,42,9.8,10,0.00
2,10248,72,34.8,5,0.00
3,10249,14,18.6,9,0.00
4,10249,51,42.4,40,0.00
5,10250,41,7.7,10,0.00
6,10250,51,42.4,35,0.15
7,10250,65,$16.8,15,0.15
8,10251,22,$16.8,6,0.05
9,10251,57,15.6,15,0.05


In [12]:
r_order_details.info()

<class 'pandas.DataFrame'>
RangeIndex: 2327 entries, 0 to 2326
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   orderID    2327 non-null   int64  
 1   productID  2327 non-null   str    
 2   unitPrice  2327 non-null   str    
 3   quantity   2327 non-null   str    
 4   discount   2327 non-null   float64
dtypes: float64(1), int64(1), str(3)
memory usage: 91.0 KB


In [13]:
# Initial audit- Duplicate Rows, Missing Values, Negative Values.
print(f"Duplicate rows: {r_order_details.duplicated().sum()}")
print(f"Order Details - Missing values detected as actual NaN: {r_order_details.isna().sum().sum()}")
print(f"Order Details - String 'NULL' values hidden: {(r_order_details == 'NULL').sum().sum()}")

# During initial profiling, a less-than comparison on 'quantity' threw a TypeError - the numeric column was wrongly stored as text.
numeric_quantity = pd.to_numeric(r_order_details['quantity'], errors='coerce')
print(f"Order Details - Negative quantities found: {(numeric_quantity < 0).sum()}")
has_symbols = r_order_details['unitPrice'].astype(str).str.contains(r'[^\w\s.]').any()
print(has_symbols)

Duplicate rows: 121
Order Details - Missing values detected as actual NaN: 0
Order Details - String 'NULL' values hidden: 0
Order Details - Negative quantities found: 115
True


In [14]:
import re

# Combine all cells into one single text string
all_text = "".join(r_order_details['unitPrice'].astype(str))

# Finding everything that is not a letter, number, space, or decimal point
all_symbols = re.findall(r'[^\w\s.]', all_text)

# Keep only the unique symbols
unique_symbols = set(all_symbols)

print("The symbols in your column are:", unique_symbols)

The symbols in your column are: {'$'}


In [15]:
# Strip whitespace/currency symbols from unitPrice, then convert to float
r_order_details['unitPrice'] = (
    r_order_details['unitPrice'].astype(str).str.strip().str.replace('$', '', regex=False).str.strip()
)
r_order_details['unitPrice'] = pd.to_numeric(r_order_details['unitPrice'], errors='coerce').astype(float)

# Convert accidental negative quantities to positive
r_order_details['quantity'] = pd.to_numeric(r_order_details['quantity'], errors='coerce').abs().astype(int)

# Restore ID and discount types
for col in ['orderID', 'productID']:
    r_order_details[col] = pd.to_numeric(r_order_details[col], errors='coerce').astype(int)
r_order_details['discount'] = pd.to_numeric(r_order_details['discount'], errors='coerce').astype(float)

# Fix drop duplicate line items, clean currency-formatted unitPrice , fix negative quantities, restore ID types
print(f"Duplicate rows: {r_order_details.duplicated().sum()}")
duplicate_count = r_order_details.duplicated().sum()
r_order_details = r_order_details.drop_duplicates().reset_index(drop=True)
print(f"  --> Removed {duplicate_count} duplicate rows from Order Details.")

print("Order Details table cleaned successfully.")
r_order_details.head()

Duplicate rows: 172
  --> Removed 172 duplicate rows from Order Details.
Order Details table cleaned successfully.


,orderID,productID,unitPrice,quantity,discount
0,10248,11,14.0,12,0.0
1,10248,42,9.8,10,0.0
2,10248,72,34.8,5,0.0
3,10249,14,18.6,9,0.0
4,10249,51,42.4,40,0.0


---
## 5. Orders Table

In [16]:
r_orders.head()

,orderID,customerID,employeeID,orderDate,requiredDate,shippedDate,shipperID,freight
0,10248,VINET,5,2022-07-04,2022-08-01,2022-07-16,3,32.38
1,10249,TOMSP,6,2022-07-05,2022-08-16,2022-07-10,1,11.61
2,10250,HANAR,4,2022-07-08,2022-08-05,2022-07-12,2,65.83
3,10251,VICTE,3,2022-07-08,2022-08-05,2022-07-15,1,41.34
4,10252,SUPRD,4,2022-07-09,2022-08-06,2022-07-11,2,51.30


In [17]:
r_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 830 entries, 0 to 829
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   orderID       830 non-null    int64         
 1   customerID    830 non-null    str           
 2   employeeID    830 non-null    str           
 3   orderDate     830 non-null    datetime64[us]
 4   requiredDate  830 non-null    datetime64[us]
 5   shippedDate   809 non-null    datetime64[us]
 6   shipperID     830 non-null    str           
 7   freight       830 non-null    float64       
dtypes: datetime64[us](3), float64(1), int64(1), str(3)
memory usage: 52.0 KB


In [18]:
# Initial audit -  Checking Duplicates, Missing Values.
print(f"Duplicate rows - :{r_orders.duplicated().sum()}")
print(f"order - String 'NULL' values hidden : {(r_orders == 'NULL').sum().sum()}")
print(f"order - Sum of Missing Values in the Table: {r_orders.isnull().sum().sum()}")

Duplicate rows - :0
order - String 'NULL' values hidden : 0
order - Sum of Missing Values in the Table: 21


In [19]:
# Checking missing value in each columns
print(f"order - Sum of Missing Values in the Table:\n{r_orders.isnull().sum()}")

order - Sum of Missing Values in the Table:
orderID          0
customerID       0
employeeID       0
orderDate        0
requiredDate     0
shippedDate     21
shipperID        0
freight          0
dtype: int64


In [20]:
# Fix restore ID types , parse dates, and fix missing dates.
for col in ['customerID']:
    r_orders[col] = r_orders[col].astype(str).str.strip()
for col in ['orderID', 'employeeID', 'shipperID']:
    r_orders[col] = pd.to_numeric(r_orders[col], errors='coerce').astype(int)

# errors='coerce' turns unparseable date strings into proper NaT (Not a Time) markers
for col in ['orderDate', 'requiredDate', 'shippedDate']:
    r_orders[col] = pd.to_datetime(r_orders[col], errors='coerce')

# Flag and reset shipments that supposedly happened before the order was placed
missing_dates = r_orders['shippedDate'] < r_orders['orderDate']
anomaly_count = missing_dates.sum()
r_orders.loc[missing_dates, 'shippedDate'] = pd.NaT

print(f"  --> Identified and neutralized {anomaly_count} date errors.")
print("Orders table cleaned successfully.")

  --> Identified and neutralized 124 date errors.
Orders table cleaned successfully.


---
## 6. Products Table

In [21]:
r_products.head()

,productID,productName,categoryID,categoryName,quantityPerUnit,unitPrice,discontinued,categoryID2
0,1,Spiced Chai Tea,1,Beverages,10 boxes x 20 bags,18.00,0,1
1,2,Import Lager Beer,1,Beverages,24 - 12 oz bottles,19.00,0,1
2,3,Flavored Baking Syrup,2,Condiments,12 - 550 ml bottles,10.00,0,2
3,4,Seasoning Blend,2,Condiments,48 - 6 oz jars,22.00,0,2
4,5,Dry Gumbo Base Mix (Soup & Stew Mix),2,Condiments,36 boxes,21.35,1,2


In [22]:
r_products.info()

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   productID        77 non-null     str    
 1   productName      77 non-null     str    
 2   categoryID       77 non-null     str    
 3   categoryName     77 non-null     str    
 4   quantityPerUnit  77 non-null     str    
 5   unitPrice        77 non-null     float64
 6   discontinued     77 non-null     str    
 7   categoryID2      77 non-null     str    
dtypes: float64(1), str(7)
memory usage: 4.9 KB


In [23]:
# Initial audit - Checking Duplicates, Missing Values , boolean formats.
print(f"Duplicate rows: {r_products.duplicated().sum()}")
print(f"Products - Missing values detected as actual NaN: {r_products.isna().sum().sum()}")
print(f"Products - String 'NULL' values hidden: {(r_products == 'NULL').sum().sum()}")
print(f"Products - Non-standard boolean formats in 'discontinued': {r_products['discontinued'].isin(['yes', 'no', 'True', 'False']).sum()}")

Duplicate rows: 0
Products - Missing values detected as actual NaN: 0
Products - String 'NULL' values hidden: 0
Products - Non-standard boolean formats in 'discontinued': 11


In [24]:
# Fix strip text fields,  restore numeric types, standardize boolean flag
for col in ['productName', 'quantityPerUnit']:
    r_products[col] = r_products[col].astype(str).str.strip()

r_products['unitPrice'] = pd.to_numeric(r_products['unitPrice'], errors='coerce').astype(float)
r_products['categoryID'] = pd.to_numeric(r_products['categoryID'], errors='coerce').astype(int)

# Standardize inconsistent boolean/text values down to a clean binary flag
boolean_map = {
    'True': 1, '1': 1, 1: 1, 'yes': 1, 'Yes': 1,
    'False': 0, '0': 0, 0: 0, 'no': 0, 'No': 0,
}
r_products['discontinued'] = (
    r_products['discontinued'].astype(str).str.strip().map(boolean_map).fillna(0).astype(int)
)

print("Products table cleaned successfully.")
r_products.head()

Products table cleaned successfully.


,productID,productName,categoryID,categoryName,quantityPerUnit,unitPrice,discontinued,categoryID2
0,1,Spiced Chai Tea,1,Beverages,10 boxes x 20 bags,18.00,0,1
1,2,Import Lager Beer,1,Beverages,24 - 12 oz bottles,19.00,0,1
2,3,Flavored Baking Syrup,2,Condiments,12 - 550 ml bottles,10.00,0,2
3,4,Seasoning Blend,2,Condiments,48 - 6 oz jars,22.00,0,2
4,5,Dry Gumbo Base Mix (Soup & Stew Mix),2,Condiments,36 boxes,21.35,1,2


---
## 7. Export Cleaned Data to SQL Server

In [25]:
r_customers.to_sql(name="Customers_Clean_Data", con=engine, if_exists="replace", index=False)
r_employees.to_sql(name="Employees_Clean_Data", con=engine, if_exists="replace", index=False)
r_order_details.to_sql(name="Orders_Details_Clean_Data", con=engine, if_exists="replace", index=False)
r_orders.to_sql(name="Orders_Clean_Data", con=engine, if_exists="replace", index=False)
r_products.to_sql(name="Products_Clean_Data", con=engine, if_exists="replace", index=False)

print("Cleaned tables exported to Microsoft SQL Server")

Cleaned tables exported to Microsoft SQL Server


---
## 8. Export Cleaned Data to Project Folder .

In [26]:
r_customers.to_csv(r"C:\Users\Yash Sonawane\Desktop\End-to-End Sales Analytics Project\3.Clean Production Data\Customers_Clean_Data.csv", index=False)
r_employees.to_csv(r"C:\Users\Yash Sonawane\Desktop\End-to-End Sales Analytics Project\3.Clean Production Data\Employees_Clean_Data.csv", index=False)
r_order_details.to_csv(r"C:\Users\Yash Sonawane\Desktop\End-to-End Sales Analytics Project\3.Clean Production Data\Orders_Details_Clean_Data.csv", index=False)
r_orders.to_csv(r"C:\Users\Yash Sonawane\Desktop\End-to-End Sales Analytics Project\3.Clean Production Data\Orders_Clean_Data.csv", index=False)
r_products.to_csv(r"C:\Users\Yash Sonawane\Desktop\End-to-End Sales Analytics Project\3.Clean Production Data\Products_Clean_Data.csv", index=False)

print("Cleaned tables exported")

Cleaned tables exported
